## Netflix Business Insights Report
Analyze the Netflix dataset and generate key business insights.

In [1]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("../../data/processed/netflix_cleaned.csv")
df.shape

(8790, 10)

In [2]:
df

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,2021-09-25,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,2021-09-24,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,2021-09-22,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,2021-09-24,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"
...,...,...,...,...,...,...,...,...,...,...
8785,s8797,TV Show,Yunus Emre,Not Given,Turkey,2017-01-17,2016,TV-PG,2 Seasons,"International TV Shows, TV Dramas"
8786,s8798,TV Show,Zak Storm,Not Given,United States,2018-09-13,2016,TV-Y7,3 Seasons,Kids' TV
8787,s8801,TV Show,Zindagi Gulzar Hai,Not Given,Pakistan,2016-12-15,2012,TV-PG,1 Season,"International TV Shows, Romantic TV Shows, TV ..."
8788,s8784,TV Show,Yoko,Not Given,Pakistan,2018-06-23,2016,TV-Y,1 Season,Kids' TV


In [3]:
df["type"].value_counts()

type
Movie      6126
TV Show    2664
Name: count, dtype: int64

In [4]:
fig = px.pie(
    df, names="type", hole=0.5,
    color_discrete_sequence=px.colors.sequential.RdBu,
    template="plotly_dark", title="Netflix Content Composition",
    height=450, width=650
)

fig.update_traces(textposition="inside", textinfo="percent+label")
fig.write_html("../outputs/business_content_composition.html")
fig.show()

In [5]:
countries = df["country"].dropna().str.split(",").explode().str.strip()
top_countries = countries[countries != "Not Given"].value_counts().head(10).reset_index()
top_countries.columns = ["Country", "Titles"]

fig = px.bar(
    top_countries.sort_values("Titles"),
    x="Titles", y="Country", orientation="h",
    color="Titles",
    color_continuous_scale=px.colors.sequential.RdBu,
    template="plotly_dark",
    title="Top 10 Countries by Netflix Content",
    height=500, width=800
)

fig.write_html("../outputs/business_top_countries.html")
fig.show()

In [6]:
ratings = df["rating"].value_counts().head(10).reset_index()
ratings.columns = ["Rating", "Titles"]

fig = px.bar(
    ratings, x="Rating", y="Titles",
    color="Titles",
    color_continuous_scale=px.colors.sequential.RdBu,
    template="plotly_dark",
    title="Top 10 Netflix Content Ratings",
    height=500, width=800
)

fig.write_html("../outputs/business_rating_analysis.html")
fig.show()

In [7]:
year_counts = df.groupby("release_year").size().reset_index(name="Titles")

fig = px.line(
    year_counts, x="release_year", y="Titles",
    markers=True,
    color_discrete_sequence=px.colors.sequential.RdBu,
    template="plotly_dark",
    title="Netflix Historical Content Trends",
    height=500, width=850
)

fig.write_html("../outputs/business_release_trends.html")
fig.show()


In [8]:
genres = df["listed_in"].dropna().str.split(",").explode().str.strip()
top_genres = genres.value_counts().head(10).reset_index()
top_genres.columns = ["Genre", "Titles"]

fig = px.bar(
    top_genres.sort_values("Titles"),
    x="Titles", y="Genre", orientation="h",
    color="Titles",
    color_continuous_scale=px.colors.sequential.RdBu,
    template="plotly_dark",
    title="Top 10 Netflix Genres",
    height=500, width=800
)

fig.write_html("../outputs/business_top_genres.html")
fig.show()


In [9]:
rating_type = df.groupby(["rating", "type"]).size().reset_index(name="Titles")

fig = px.bar(
    rating_type, x="rating", y="Titles", color="type",
    barmode="stack",
    color_discrete_sequence=px.colors.sequential.RdBu,
    template="plotly_dark",
    title="Netflix Ratings by Content Type",
    height=500, width=850
)

fig.write_html("../outputs/business_ratings_by_type.html")
fig.show()

In [10]:
df["Decade"] = (df["release_year"] // 10) * 10
decades = df["Decade"].value_counts().sort_index().reset_index()
decades.columns = ["Decade", "Titles"]

fig = px.bar(
    decades, x="Decade", y="Titles",
    color="Titles",
    color_continuous_scale=px.colors.sequential.RdBu,
    template="plotly_dark",
    title="Netflix Content by Decade",
    height=450, width=800
)

fig.write_html("../outputs/business_content_by_decade.html")
fig.show()


In [11]:
fig.update_coloraxes(colorscale=px.colors.sequential.Plasma)
fig.write_html("../outputs/business_content_by_decade.html")
fig.show()

In [12]:
df["date_added"] = pd.to_datetime(df["date_added"], errors="coerce")
added_year = df["date_added"].dt.year.value_counts().sort_index().reset_index()
added_year.columns = ["Year", "Titles"]

fig = px.line(
    added_year, x="Year", y="Titles",
    markers=True,
    color_discrete_sequence=px.colors.sequential.Plasma,
    template="plotly_dark",
    title="Netflix Content Added Over Time",
    height=450, width=800
)

fig.write_html("../outputs/business_content_added_trend.html")
fig.show()

In [13]:
df["Month"] = df["date_added"].dt.month

monthly = df["Month"].value_counts().sort_index().reset_index()
monthly.columns = ["Month", "Titles"]

fig = px.bar(
    monthly, x="Month", y="Titles",
    color="Titles",
    color_continuous_scale=px.colors.sequential.Plasma,
    template="plotly_dark",
    title="Monthly Netflix Content Additions",
    height=450, width=800
)

fig.write_html("../outputs/business_monthly_additions.html")
fig.show()

In [14]:
df["duration_num"] = pd.to_numeric(
    df[df["type"] == "Movie"]["duration"].str.extract(r"(\d+)")[0],
    errors="coerce"
)

fig = px.histogram(
    df, x="duration_num",
    color_discrete_sequence=px.colors.sequential.Plasma,
    template="plotly_dark",
    title="Netflix Movie Duration Distribution",
    labels={"duration_num": "Duration (Minutes)"},
    height=450, width=800
)

fig.write_html("../outputs/business_movie_duration.html")
fig.show()

In [16]:
movie_data = df[df["type"] == "Movie"].copy()
movie_data["Minutes"] = pd.to_numeric(
    movie_data["duration"].str.extract(r"(\d+)")[0],
    errors="coerce"
)

fig = px.box(
    movie_data, x="rating", y="Minutes", color="rating",
    color_discrete_sequence=px.colors.sequential.Plasma,
    template="plotly_dark",
    title="Movie Duration by Content Rating",
    height=500, width=850
)

fig.write_html("../outputs/business_duration_by_rating.html")
fig.show()

In [17]:
print("Total Titles:", len(df))
print("Movies:", (df["type"] == "Movie").sum())
print("TV Shows:", (df["type"] == "TV Show").sum())
print("Unique Ratings:", df["rating"].nunique())
print("Release Year Range:", df["release_year"].min(), "-", df["release_year"].max())

Total Titles: 8790
Movies: 6126
TV Shows: 2664
Unique Ratings: 14
Release Year Range: 1925 - 2021


In [18]:
print("Top Content Type:", df["type"].mode()[0])
print("Top Country:", countries[countries != "Not Given"].value_counts().idxmax())
print("Top Genre:", df["listed_in"].dropna().str.split(",").explode().str.strip().value_counts().idxmax())
print("Most Common Rating:", df["rating"].mode()[0])
print("Peak Release Year:", df["release_year"].value_counts().idxmax())

Top Content Type: Movie
Top Country: United States
Top Genre: International Movies
Most Common Rating: TV-MA
Peak Release Year: 2018


In [19]:
insights = pd.DataFrame({
    "Metric": ["Total Titles", "Movies", "TV Shows", "Top Country",
               "Top Genre", "Most Common Rating", "Peak Release Year"],
    "Finding": [8790, 6126, 2664, "United States",
                "International Movies", "TV-MA", 2018]
})

insights.to_csv("../outputs/business_insights.csv", index=False)
print("Business insights report saved!")

Business insights report saved!


In [20]:
report = f"""
NETFLIX BUSINESS INSIGHTS REPORT
================================

Total Titles: {len(df)}
Movies: {(df["type"] == "Movie").sum()}
TV Shows: {(df["type"] == "TV Show").sum()}
Top Country: {countries[countries != "Not Given"].value_counts().idxmax()}
Most Common Rating: {df["rating"].mode()[0]}
Peak Release Year: {df["release_year"].value_counts().idxmax()}

Conclusion:
Movies dominate the dataset, the United States leads
country-wise title counts, and 2018 is the peak release year.
"""

with open("../outputs/business_insights_report.txt", "w", encoding="utf-8") as file:
    file.write(report)

print("Business report saved successfully!")

Business report saved successfully!


In [21]:
from pathlib import Path

project = Path("../../")
charts = list(project.glob("Task_*/outputs/*.html"))

print("Total interactive charts:", len(charts))

Total interactive charts: 35


In [22]:
from pathlib import Path

p = Path("../../requirements.txt")
text = p.read_text()

if "plotly" not in text.lower():
    p.write_text(text.rstrip() + "\nplotly\n")

print("Requirements updated!")

Requirements updated!


In [23]:
from pathlib import Path

project = Path("../../")

for file in [
    "README.md",
    "requirements.txt",
    "data/raw/Dataset.csv",
    "data/processed/netflix_cleaned.csv"
]:
    print(file, "✅" if (project / file).exists() else "❌")

print("Interactive charts:",
      len(list(project.glob("Task_*/outputs/*.html"))))

README.md ✅
requirements.txt ✅
data/raw/Dataset.csv ✅
data/processed/netflix_cleaned.csv ✅
Interactive charts: 35
